# KNN Model - PharmShed Super Dataset

**Model:** K-Nearest Neighbors (sklearn) with Hassanat distance metric
**Task:** Multi-class classification — predict which of 218 classes a person is prescribed (217 drugs + "no prescriptions")
**Features:** Demographics (Age, Sex, Family_income, Insurance_coverage, Race_ethnicity) + Prescription (Quantity, Form, Strength, Day_Supply)
**Split strategy:** StratifiedGroupKFold (5-fold CV, shuffle=True, random_state=42), grouped by Person_ID to prevent data leakage
**Missing values:** Prescription NaNs for "no prescription" rows filled with -1 sentinel; Age, Strength, and Day_Supply imputed within each fold using hierarchical median imputation
**Preprocessing:** Categorical features one-hot encoded; numeric features scaled with RobustScaler before Hassanat distance computation
**Metrics:** Accuracy, Cohen's Kappa, MCC, macro/micro averaged Precision, Recall, F1, F2
**Output:** Saves `knn_super_proba_2022.csv` — probability vector over 218 classes per observation, used as input to the ensemble model

## Key Design Decisions

**Why Hassanat distance?**
Our dataset has 218 classes with severe class imbalance — many rare drugs have very few records.
Euclidean distance is dominated by common drugs simply because they occupy more of the feature space.
Hassanat distance is bounded between 0 and 1 regardless of feature scale, making it robust to outliers
and rare records. It was specifically chosen for this problem because it performs well when classes
are sparse and unevenly distributed. Implemented as a custom metric passed to KNeighborsClassifier
via the metric parameter.

**Why -1 for "no prescription" NaNs?**
The 97,497 "no prescriptions" rows have NaN for Quantity, Form, Strength, and Day_Supply because those fields are structurally absent — not unknown.
Filling with -1 preserves this as a meaningful signal. After RobustScaler, -1 maps to a distinct region in feature space away from real prescription values.

**Why impute Age, Strength, and Day_Supply inside each fold?**
Age, Strength, and Day_Supply have real missing values (not structural absences) that require imputation.
Imputation is applied within each fold — on train and val separately — to prevent data leakage.
Age uses a hierarchical median strategy (household → income/insurance group → year → global).
Strength and Day_Supply use median imputation grouped by year and drug.

**Why RobustScaler even with Hassanat?**
Hassanat distance is naturally bounded 0–1 per feature, but RobustScaler is still applied before
computing it. This ensures all features contribute equally when Hassanat aggregates distances
across multiple dimensions. Without scaling, high-range features like Family_income would still
have disproportionate influence on the aggregated distance. RobustScaler is preferred over
StandardScaler because it uses median and IQR, making it more robust to the outliers produced
by our severely imbalanced class distribution.

**Why one-hot encode categoricals?**
KNN cannot natively interpret ordinal codes for unordered categories. One-hot encoding converts
each category to a binary feature so Hassanat distance is computed meaningfully across
categorical dimensions.

**Note on compute:** KNN with 900k+ rows is expensive. Hassanat is implemented as a custom
Python function which is slower than built-in sklearn metrics. If runtime is prohibitive,
consider running CV on a stratified subsample and retraining the final model on the full dataset.

In [ ]:
# Install required libraries.
# scikit-learn: KNN model, preprocessing, CV, metrics.
# permetrics: macro/micro F2 score and other evaluation metrics.
!pip install scikit-learn permetrics

In [ ]:
# Import all required libraries.
import pandas as pd
import numpy as np
import sklearn
import joblib
from sklearn.preprocessing import LabelEncoder, RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, matthews_corrcoef,
    classification_report
)
from sklearn.utils import shuffle
from permetrics import ClassificationMetric
import warnings
warnings.filterwarnings('ignore')

# Confirm versions for reproducibility
print('scikit-learn version:', sklearn.__version__)
print('pandas version:', pd.__version__)
print('numpy version:', np.__version__)

# OneHotEncoder parameter name changed from 'sparse' to 'sparse_output' in sklearn 1.2.
# This flag ensures the notebook runs on both older and newer sklearn versions.
from packaging import version
OHE_SPARSE_KWARG = 'sparse_output' if version.parse(sklearn.__version__) >= version.parse('1.2') else 'sparse'

# Hassanat distance function.
# Not available in sklearn — must be defined manually and passed as a custom metric.
# Formula: for two scalar values a and b:
#   if min(a, b) >= 0: d = 1 - (1 + min(a,b)) / (1 + max(a,b))
#   else:              d = 1 - (1 + min(a,b) + abs(min(a,b))) / (1 + max(a,b) + abs(min(a,b)))
# For two vectors, the total distance is the sum of per-feature distances.
# Bounded between 0 and 1 per feature — robust to outliers and rare records.
# Chosen because it handles sparse, imbalanced class distributions better than Euclidean.
def hassanat_distance(a, b):
    total = 0.0
    for x, y in zip(a, b):
        min_val = min(x, y)
        max_val = max(x, y)
        if min_val >= 0:
            total += 1 - (1 + min_val) / (1 + max_val)
        else:
            shift = abs(min_val)
            total += 1 - (1 + min_val + shift) / (1 + max_val + shift)
    return total

print('Hassanat distance function defined.')

In [ ]:
# Load the super integrated dataset (2014-2021).
# The super dataset contains demographics + prescription features + Person_ID.

DATA_DIR = './'  # change this to your data path if needed

super_df = pd.read_csv(f'{DATA_DIR}super_integrated_data.csv')

if 'Unnamed: 0' in super_df.columns:
    super_df = super_df.drop(columns=['Unnamed: 0'])

print('Super dataset shape:', super_df.shape)
print('\nSuper dataset columns:', super_df.columns.tolist())
print('\nMissing values:')
print(super_df.isnull().sum())

In [ ]:
# Verify Person_ID is present in the dataset.
# Person_ID is used only for StratifiedGroupKFold grouping — not a model feature.
assert 'Person_ID' in super_df.columns, \
    "ERROR: Person_ID not found in dataset — check super_integrated_data.csv."
assert super_df['Person_ID'].isnull().sum() == 0, \
    "ERROR: Missing Person_IDs — check super_integrated_data.csv."

print('Person_ID verified.')
print('Unique persons:', super_df['Person_ID'].nunique())

In [ ]:
# Prescription NaN handling — fill with -1 sentinel.
# The 97,497 no-prescription rows have NaN for Quantity/Strength/Day_Supply/Form
# because those fields are structurally absent (not missing at random).
# Filling with -1 preserves this as a meaningful signal distinct from real values.
# After RobustScaler, -1 maps to a distinct region in feature space.
#
# Age, Strength, and Day_Supply real missing values are handled inside the CV loop
# using hierarchical median imputation on the train split only — see the CV cell below.

numeric_prescription_cols = ['Quantity', 'Strength', 'Day_Supply']
super_df[numeric_prescription_cols] = super_df[numeric_prescription_cols].fillna(-1)

# Form is a categorical string column (values like TABS, ORAL, CAPS).
# Fill with string '-1' so it is treated as a distinct 'no prescription' category
# by the OneHotEncoder later.
super_df['Form'] = super_df['Form'].fillna('-1')

print('Missing values after prescription fill:')
print(super_df[['Quantity', 'Form', 'Strength', 'Day_Supply']].isnull().sum())
print('\nRemaining demographic NaNs (will be imputed inside CV loop):')
print(super_df[['Age', 'Family_income']].isnull().sum())

# Hard stop if prescription columns still have NaNs after fill.
# These must be zero — any remaining NaN would silently corrupt distance computation.
assert super_df[numeric_prescription_cols + ['Form']].isnull().sum().sum() == 0, \
    "ERROR: Prescription NaNs remain after fill — check fillna logic before proceeding."

print('\nAll prescription NaNs filled successfully.')

In [ ]:
# Imputation functions for Age, Strength, and Day_Supply.
# Applied within each fold to prevent data leakage.
# Consistent with TabICL and all other base models.

def impute_age(df):
    df = df.copy()
    df['income_bracket'] = (
        df.groupby('Year')['Family_income']
        .transform(lambda x: pd.qcut(x, 4, labels=False, duplicates='drop') + 1)
    )
    hh_meds    = df.groupby(['Year', 'Household_ID'])['Age'].transform('median')
    grp_meds   = df.groupby(['Year', 'income_bracket', 'Insurance_coverage'])['Age'].transform('median')
    yr_meds    = df.groupby('Year')['Age'].transform('median')
    global_med = df['Age'].median()
    df['Age']  = df['Age'].fillna(hh_meds)
    df['Age']  = df['Age'].fillna(grp_meds)
    df['Age']  = df['Age'].fillna(yr_meds)
    df['Age']  = df['Age'].fillna(global_med)
    df.drop(columns=['income_bracket'], inplace=True)
    return df

def impute_strength(df):
    df = df.copy()
    median_strength = df.groupby(['Year', 'Drug'])['Strength'].transform('median')
    df['Strength']  = df['Strength'].fillna(median_strength)
    return df

def impute_day_supply(df):
    df = df.copy()
    median_day_supply = df.groupby(['Year', 'Drug'])['Day_Supply'].transform('median')
    df['Day_Supply']  = df['Day_Supply'].fillna(median_day_supply)
    return df

In [ ]:
# Exploratory check before modeling.
# Verify class count, drug distribution, and person count.
print('Unique persons:', super_df['Person_ID'].nunique())
print('Unique drugs:', super_df['Drug'].nunique())
print('No prescription rows:', (super_df['Drug'] == 'no prescriptions').sum())
print('Actual drug rows:', (super_df['Drug'] != 'no prescriptions').sum())

print('\nTop 10 most prescribed drugs:')
print(super_df['Drug'].value_counts().head(10))

print('\nBottom 5 rarest drugs:')
print(super_df['Drug'].value_counts().tail(5))

# Check class imbalance ratio — max count / min count.
# High ratio confirms severe imbalance and justifies Hassanat distance
# and distance-weighted voting in KNN.
counts = super_df['Drug'].value_counts()
print(f'\nMost common drug count:  {counts.max():,}')
print(f'Rarest drug count:       {counts.min():,}')
print(f'Imbalance ratio:         {counts.max() / counts.min():.1f}x')

# Hard stop if drug count is not 218 (217 drugs + "no prescriptions").
assert super_df['Drug'].nunique() == 218, \
    f"ERROR: Expected 218 drug classes, found {super_df['Drug'].nunique()}."

print('\nDrug class count confirmed: 218 (217 drugs + no prescriptions).')

In [ ]:
# Define feature columns, categorical columns, numeric columns, and target.
#
# Categorical features are one-hot encoded (KNN cannot natively handle unordered
# categorical values — ordinal codes would imply false ordinal relationships).
#
# Numeric features are scaled with RobustScaler inside the CV loop
# to prevent scale leakage across folds. RobustScaler is used over StandardScaler
# because it is less sensitive to outliers due to our severe class imbalance.
#
# LabelEncoder is fit once on the full dataset before the CV loop.
# Fitting inside the loop would produce different integer mappings per fold,
# making fold results incomparable and breaking per-drug recall aggregation.

feature_cols     = ['Age', 'Sex', 'Family_income', 'Insurance_coverage',
                    'Race_ethnicity', 'Quantity', 'Form', 'Strength', 'Day_Supply']
categorical_cols = ['Sex', 'Insurance_coverage', 'Race_ethnicity', 'Form']
numeric_cols     = ['Age', 'Family_income', 'Quantity', 'Strength', 'Day_Supply']
target_col       = 'Drug'

le = LabelEncoder()
super_df['Drug_encoded'] = le.fit_transform(super_df[target_col])

print('Unique classes:', len(le.classes_))
print('Feature cols:', feature_cols)
print('Categorical cols:', categorical_cols)
print('Numeric cols:', numeric_cols)
print('\nSample drug to integer mapping (first 5):')
for i, drug in enumerate(le.classes_[:5]):
    print(f'  {drug} -> {i}')

# Hard stop if any defined column is missing from the dataset.
# Catches typos in column names or dataset changes before the CV loop starts.
missing_cols = [c for c in feature_cols + [target_col] if c not in super_df.columns]
assert len(missing_cols) == 0, \
    f"ERROR: These columns are missing from the dataset: {missing_cols}"

# Save the LabelEncoder so it can be reloaded without rerunning this notebook.
# Required for ensemble model — all models must use identical drug-to-integer mappings.
joblib.dump(le, 'knn_super_label_encoder.joblib')
print('\nLabelEncoder saved to knn_super_label_encoder.joblib')

In [ ]:
# Define the preprocessing pipeline.
#
# Numeric pipeline:
#   RobustScaler — scales using median and IQR instead of mean and std.
#   More robust to outliers than StandardScaler, which is important given
#   our severe class imbalance where rare drug classes can produce outlier values.
#   Applied before Hassanat distance computation to ensure all features
#   contribute equally when distances are aggregated across dimensions.
#   Fitted on train data only inside each fold to prevent scale leakage.
#   Note: Age, Strength, and Day_Supply are imputed before this pipeline runs
#   using hierarchical median imputation functions — SimpleImputer is not used.
#
# Categorical pipeline:
#   OneHotEncoder — converts categories to binary indicator columns.
#   handle_unknown='ignore' safely handles rare categories in val
#   not seen during training.
#   sparse_output=False returns a dense array (required for KNN with
#   a custom distance metric like Hassanat).
#
# NOTE: The preprocessor is fit on train data only inside each fold.
# It is NOT fit here — fitting here would leak val statistics into
# scaler medians/IQRs, inflating CV performance.

numeric_transformer = Pipeline(steps=[
    ('scaler', RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(**{OHE_SPARSE_KWARG: False}, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

print('Preprocessor defined.')
print('  Numeric pipeline:     RobustScaler')
print('  Categorical pipeline: OneHotEncoder')
print('  Note: preprocessor is not fitted here — fitted inside each CV fold on train split only.')

In [ ]:
# 5-fold stratified group cross-validation.
#
# StratifiedGroupKFold ensures:
#   1. Each fold has similar drug class distribution (stratified)
#   2. All records for the same person stay in the same fold (grouped)
#      preventing the model from seeing the same person in both train and val
#
# For each fold:
#   1. Split by Person_ID groups and drug label stratification
#   2. Impute Age, Strength, Day_Supply within each fold (fit on train, apply to val)
#   3. Fit RobustScaler and OneHotEncoder on train split only — apply to both train and val
#   4. Train KNN with Hassanat distance on preprocessed train fold
#   5. Predict on preprocessed val fold
#   6. Compute and store all metrics + per-drug recall

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

X      = super_df[feature_cols].copy()
y      = super_df['Drug_encoded'].values
groups = super_df['Person_ID'].values

fold_results          = []
per_drug_recall_folds = []

for fold_num, (train_idx, val_idx) in enumerate(sgkf.split(X, y, groups), start=1):
    print(f'\n{"="*50}')
    print(f'FOLD {fold_num}/5')
    print(f'{"="*50}')

    train_fold = super_df.iloc[train_idx].copy()
    val_fold   = super_df.iloc[val_idx].copy()

    # Impute Age, Strength, Day_Supply within each fold.
    # Imputation is applied on train and val separately to prevent data leakage.
    train_fold = impute_age(train_fold)
    val_fold   = impute_age(val_fold)

    train_fold = impute_strength(train_fold)
    val_fold   = impute_strength(val_fold)

    train_fold = impute_day_supply(train_fold)
    val_fold   = impute_day_supply(val_fold)

    X_train_fold = train_fold[feature_cols].copy()
    X_val_fold   = val_fold[feature_cols].copy()
    y_train_fold = train_fold['Drug_encoded'].values
    y_val_fold   = val_fold['Drug_encoded'].values

    print(f'Train size: {len(X_train_fold):,} | Val size: {len(X_val_fold):,}')
    print(f'Unique drugs in train: {len(np.unique(y_train_fold))} | val: {len(np.unique(y_val_fold))}')

    # Fit preprocessor on train split only.
    # RobustScaler and OneHotEncoder statistics are computed from train data only —
    # no leakage from val into preprocessing.
    X_train_processed = preprocessor.fit_transform(X_train_fold)
    X_val_processed   = preprocessor.transform(X_val_fold)

    print(f'Preprocessed train shape: {X_train_processed.shape}')

    # Train KNN with Hassanat distance.
    # metric=hassanat_distance: custom function defined in Cell 2.
    #   Bounded 0-1 per feature — robust to outliers and rare records.
    #   Better than Euclidean for imbalanced 218-class problem.
    # weights='distance': closer neighbors vote more heavily — helps rare classes.
    # algorithm='ball_tree': required when using a custom distance metric.
    #   'auto' and 'kd_tree' do not support custom callables.
    # n_jobs=-1: use all CPU cores for distance computation.
    model = KNeighborsClassifier(
        n_neighbors=5,
        weights='distance',
        metric=hassanat_distance,
        algorithm='ball_tree',
        n_jobs=-1
    )
    model.fit(X_train_processed, y_train_fold)
    print(f'Fold {fold_num} training complete.')

    y_pred_fold = model.predict(X_val_processed)

    # Overall metrics.
    acc   = accuracy_score(y_val_fold, y_pred_fold)
    kappa = cohen_kappa_score(y_val_fold, y_pred_fold)
    mcc   = matthews_corrcoef(y_val_fold, y_pred_fold)

    # Macro and micro averaged metrics via permetrics.
    # Macro: average per class equally — treats rare and common drugs equally.
    # Micro: aggregate all counts — dominated by the most common drugs.
    evaluator = ClassificationMetric(y_val_fold, y_pred_fold)

    macro_precision = evaluator.precision_score(average='macro')
    micro_precision = evaluator.precision_score(average='micro')
    macro_recall    = evaluator.recall_score(average='macro')
    micro_recall    = evaluator.recall_score(average='micro')
    macro_f1        = evaluator.f1_score(average='macro')
    micro_f1        = evaluator.f1_score(average='micro')

    # F2 score weights recall twice as much as precision.
    # Higher recall is more important here because missing a drug class
    # means underestimating its wastewater load.
    macro_f2 = evaluator.fbeta_score(beta=2, average='macro')
    micro_f2 = evaluator.fbeta_score(beta=2, average='micro')

    fold_results.append({
        'fold': fold_num,
        'accuracy': acc,
        'cohen_kappa': kappa,
        'mcc': mcc,
        'macro_precision': macro_precision,
        'micro_precision': micro_precision,
        'macro_recall': macro_recall,
        'micro_recall': micro_recall,
        'macro_f1': macro_f1,
        'micro_f1': micro_f1,
        'macro_f2': macro_f2,
        'micro_f2': micro_f2,
    })

    # Per-drug recall for this fold.
    # Used for ensemble model selection — the ensemble picks the best model per drug.
    report = classification_report(
        y_val_fold, y_pred_fold,
        labels=np.arange(len(le.classes_)),
        target_names=le.classes_,
        output_dict=True,
        zero_division=0
    )
    drug_recalls         = {drug: report[drug]['recall'] for drug in le.classes_ if drug in report}
    drug_recalls['fold'] = fold_num
    per_drug_recall_folds.append(drug_recalls)

    print(f'Fold {fold_num} results:')
    print(f'  Accuracy:      {acc:.4f}')
    print(f'  Cohen Kappa:   {kappa:.4f}')
    print(f'  MCC:           {mcc:.4f}')
    print(f'  Macro Recall:  {macro_recall:.4f}')
    print(f'  Micro Recall:  {micro_recall:.4f}')
    print(f'  Macro F2:      {macro_f2:.4f}')

print('\n' + '='*50)
print('ALL FOLDS COMPLETE')
print('='*50)

In [ ]:
# Summarize cross-validation results across all 5 folds.
# Report mean and standard deviation for each metric.
# Standard deviation shows how stable the model is across different data splits.
results_df = pd.DataFrame(fold_results)

print('CV RESULTS — MEAN +/- STD ACROSS 5 FOLDS')
print('='*55)
metric_cols = [c for c in results_df.columns if c != 'fold']
for col in metric_cols:
    mean = results_df[col].mean()
    std  = results_df[col].std()
    print(f'  {col:<25}: {mean:.4f} +/- {std:.4f}')

results_df.to_csv('knn_super_cv_results.csv', index=False)
print('\nCV results saved to knn_super_cv_results.csv')

# Quick sanity check — all 5 folds must be present before summarizing.
# If the CV loop crashed mid-run, this catches it before saving incomplete results.
assert len(results_df) == 5, \
    f"ERROR: Expected 5 fold results, found {len(results_df)} — CV may not have completed."

print('All 5 folds confirmed complete.')

In [ ]:
# Compute average per-drug recall across all 5 folds.
# This is the key output for ensemble model selection.
# The ensemble picks whichever base model has the highest recall for each drug.
per_drug_df = pd.DataFrame(per_drug_recall_folds)
drug_cols   = [c for c in per_drug_df.columns if c != 'fold']

mean_drug_recall = per_drug_df[drug_cols].mean().reset_index()
mean_drug_recall.columns = ['Drug', 'Mean_Recall_KNN_Super']
mean_drug_recall = mean_drug_recall.sort_values('Mean_Recall_KNN_Super', ascending=False)

print('Top 20 drugs by mean recall:')
print(mean_drug_recall.head(20).to_string(index=False))

print('\nBottom 20 drugs by mean recall:')
print(mean_drug_recall.tail(20).to_string(index=False))

# Summary stats — how many drugs does KNN recall at all?
# A drug is considered recalled if mean recall > 0 across folds.
# This number goes directly into the paper for model comparison.
drugs_recalled = (mean_drug_recall['Mean_Recall_KNN_Super'] > 0).sum()
print(f'\nDrugs with mean recall > 0: {drugs_recalled} / {len(mean_drug_recall)}')
print(f'Drugs with mean recall >= 0.1: {(mean_drug_recall["Mean_Recall_KNN_Super"] >= 0.1).sum()}')
print(f'Drugs with mean recall >= 0.5: {(mean_drug_recall["Mean_Recall_KNN_Super"] >= 0.5).sum()}')

mean_drug_recall.to_csv('knn_super_per_drug_recall.csv', index=False)
print('\nPer-drug recall saved to knn_super_per_drug_recall.csv')

In [ ]:
# Train final model on the full training dataset (2014-2021).
# After CV gives confidence in model performance, retrain on all available data
# to maximize signal before evaluating on the held-out 2022 validation set.
#
# The preprocessor is fit on the full training data here (no fold split).
# The fitted preprocessor is reused to transform 2022 data in the next cell.

# Impute Age, Strength, Day_Supply on full dataset before final training.
data_final = impute_age(super_df.copy())
data_final = impute_strength(data_final)
data_final = impute_day_supply(data_final)

X_final = data_final[feature_cols].copy()
y_final = data_final['Drug_encoded'].values

# Shuffle to separate refill records after imputation.
X_final, y_final = shuffle(X_final, y_final, random_state=42)
X_final = X_final.reset_index(drop=True)

print('Final training data size:', X_final.shape)
print('Number of classes:', len(le.classes_))
print('\nFitting preprocessor on full training data...')

X_final_processed = preprocessor.fit_transform(X_final)
print('Preprocessed shape:', X_final_processed.shape)

print('\nTraining final KNN model with Hassanat distance...')
final_model = KNeighborsClassifier(
    n_neighbors=5,
    weights='distance',
    metric=hassanat_distance,
    algorithm='ball_tree',
    n_jobs=-1
)
final_model.fit(X_final_processed, y_final)
print('Final model training complete.')

# Save final model and preprocessor so the notebook does not need to be
# rerun if the kernel dies. Preprocessor must be saved alongside the model
# because it must be applied identically to 2022 data in the next cell.
joblib.dump(final_model, 'knn_super_final_model.joblib')
joblib.dump(preprocessor, 'knn_super_preprocessor.joblib')
print('\nFinal model saved to knn_super_final_model.joblib')
print('Preprocessor saved to knn_super_preprocessor.joblib')

In [ ]:
# Load 2022 super dataset (held-out internal validation set).
# Must use super_data_2022.csv — not data_2022.csv —
# because the model was trained with prescription features.
#
# Preprocessing must mirror training exactly:
#   - Prescription NaNs filled with -1 for "no prescription" rows
#   - Age, Strength, Day_Supply imputed using same functions as training
#   - Preprocessor.transform() (not fit_transform) — uses training statistics

data_2022 = pd.read_csv(f'{DATA_DIR}super_data_2022.csv')
if 'Unnamed: 0' in data_2022.columns:
    data_2022 = data_2022.drop(columns=['Unnamed: 0'])

# Same sentinel filling as training — must be identical preprocessing
numeric_prescription_cols = ['Quantity', 'Strength', 'Day_Supply']
data_2022[numeric_prescription_cols] = data_2022[numeric_prescription_cols].fillna(-1)
data_2022['Form'] = data_2022['Form'].fillna('-1')

print('2022 data shape:', data_2022.shape)

# Drop any drugs in 2022 not seen during training
unseen = set(data_2022['Drug'].unique()) - set(le.classes_)
print(f'Unseen drugs in 2022 (will be dropped): {unseen}')
data_2022 = data_2022[data_2022['Drug'].isin(le.classes_)].reset_index(drop=True)
print(f'2022 rows after filtering: {len(data_2022):,}')

# Impute Age, Strength, Day_Supply — must mirror training preprocessing.
data_2022 = impute_age(data_2022)
data_2022 = impute_strength(data_2022)
data_2022 = impute_day_supply(data_2022)

X_2022 = data_2022[feature_cols].copy()

# Apply training preprocessor — transform only, do not refit.
# Uses RobustScaler statistics from 2014-2021 training data.
X_2022_processed = preprocessor.transform(X_2022)

y_2022_encoded = le.transform(data_2022['Drug'])

# Shuffle to separate refill records.
X_2022_processed, y_2022_encoded = shuffle(X_2022_processed, y_2022_encoded, random_state=42)

y_pred_2022 = final_model.predict(X_2022_processed)

# Compute validation metrics
acc_2022   = accuracy_score(y_2022_encoded, y_pred_2022)
kappa_2022 = cohen_kappa_score(y_2022_encoded, y_pred_2022)
mcc_2022   = matthews_corrcoef(y_2022_encoded, y_pred_2022)

evaluator_2022    = ClassificationMetric(y_2022_encoded, y_pred_2022)
macro_prec_2022   = evaluator_2022.precision_score(average='macro')
micro_prec_2022   = evaluator_2022.precision_score(average='micro')
macro_recall_2022 = evaluator_2022.recall_score(average='macro')
micro_recall_2022 = evaluator_2022.recall_score(average='micro')
macro_f2_2022     = evaluator_2022.fbeta_score(beta=2, average='macro')
micro_f2_2022     = evaluator_2022.fbeta_score(beta=2, average='micro')

print('\nINTERNAL VALIDATION — MEPS 2022 Results')
print(f'Accuracy:          {acc_2022:.4f}')
print(f'Cohen Kappa:       {kappa_2022:.4f}')
print(f'MCC:               {mcc_2022:.4f}')
print(f'Macro Precision:   {macro_prec_2022:.4f}')
print(f'Micro Precision:   {micro_prec_2022:.4f}')
print(f'Macro Recall:      {macro_recall_2022:.4f}')
print(f'Micro Recall:      {micro_recall_2022:.4f}')
print(f'Macro F2:          {macro_f2_2022:.4f}')
print(f'Micro F2:          {micro_f2_2022:.4f}')

# Save probability outputs for ensemble model.
# Shape: (n_observations, 218) — one row per observation, one column per drug class.
# Columns are ordered by le.classes_ — identical mapping across all base models.
# Observation_ID is included so the ensemble script can align rows across models.
proba_2022 = final_model.predict_proba(X_2022_processed)
proba_df   = pd.DataFrame(proba_2022, columns=le.classes_)
proba_df.insert(0, 'Observation_ID', data_2022['Observation_ID'].reset_index(drop=True).values)
proba_df.to_csv('knn_super_proba_2022.csv', index=False)
print(f'\nProbability output saved to knn_super_proba_2022.csv')
print(f'Shape: {proba_df.shape} — {len(data_2022):,} observations x 218 classes + Observation_ID')

In [ ]:
# Save per-drug metrics and overall validation summary for 2022.
report_2022 = classification_report(
    y_2022_encoded, y_pred_2022,
    labels=np.arange(len(le.classes_)),
    target_names=le.classes_,
    output_dict=True,
    zero_division=0
)

drug_recall_2022 = pd.DataFrame([
    {
        'Drug': drug,
        'Recall_2022': report_2022[drug]['recall'],
        'Precision_2022': report_2022[drug]['precision'],
        'F1_2022': report_2022[drug]['f1-score'],
        'Support_2022': report_2022[drug]['support']
    }
    for drug in le.classes_ if drug in report_2022
]).sort_values('Recall_2022', ascending=False)

print('Top 15 drugs by recall on 2022 data:')
print(drug_recall_2022.head(15).to_string(index=False))
print('\nBottom 15 drugs by recall on 2022 data:')
print(drug_recall_2022.tail(15).to_string(index=False))

# Summary stats — mirrors Cell 11 but on held-out 2022 data instead of CV.
# Confirms whether CV recall numbers generalise to unseen data.
drugs_recalled_2022 = (drug_recall_2022['Recall_2022'] > 0).sum()
print(f'\nDrugs recalled on 2022 data (recall > 0): {drugs_recalled_2022} / {len(drug_recall_2022)}')
print(f'Drugs with recall >= 0.1: {(drug_recall_2022["Recall_2022"] >= 0.1).sum()}')
print(f'Drugs with recall >= 0.5: {(drug_recall_2022["Recall_2022"] >= 0.5).sum()}')

drug_recall_2022.to_csv('knn_super_2022_per_drug_metrics.csv', index=False)
print('\nPer-drug 2022 metrics saved to knn_super_2022_per_drug_metrics.csv')

validation_summary = pd.DataFrame([{
    'model': 'KNN_Super',
    'dataset': 'MEPS_2022_internal_validation',
    'accuracy': acc_2022,
    'cohen_kappa': kappa_2022,
    'mcc': mcc_2022,
    'macro_precision': macro_prec_2022,
    'micro_precision': micro_prec_2022,
    'macro_recall': macro_recall_2022,
    'micro_recall': micro_recall_2022,
    'macro_f2': macro_f2_2022,
    'micro_f2': micro_f2_2022,
}])
validation_summary.to_csv('knn_super_validation_summary.csv', index=False)
print('Validation summary saved to knn_super_validation_summary.csv')

# Final output file summary — confirms everything saved correctly.
print('\n' + '='*55)
print('ALL OUTPUTS SAVED')
print('='*55)
print('  knn_super_cv_results.csv')
print('  knn_super_per_drug_recall.csv')
print('  knn_super_2022_per_drug_metrics.csv')
print('  knn_super_validation_summary.csv')
print('  knn_super_proba_2022.csv           <- ensemble input')
print('  knn_super_final_model.joblib')
print('  knn_super_preprocessor.joblib')
print('  knn_super_label_encoder.joblib')

## Output Files

| File | Contents |
|------|----------|
| `knn_super_cv_results.csv` | Mean +/- std for all metrics across 5 CV folds |
| `knn_super_per_drug_recall.csv` | Average recall per drug across 5 folds (for ensemble selection) |
| `knn_super_2022_per_drug_metrics.csv` | Per-drug recall, precision, F1 on MEPS 2022 internal validation |
| `knn_super_validation_summary.csv` | Overall validation metrics on MEPS 2022 |
| `knn_super_proba_2022.csv` | Probability vector over 218 classes per observation — input to ensemble model |
| `knn_super_final_model.joblib` | Saved KNN model with Hassanat distance |
| `knn_super_preprocessor.joblib` | Saved RobustScaler + OneHotEncoder pipeline |
| `knn_super_label_encoder.joblib` | Saved LabelEncoder for drug class mapping |

**Next step:** Pass `knn_super_proba_2022.csv` to the ensemble model (`ensemble_pharmshed.ipynb`) along with probability outputs from all other base models for soft voting ensemble construction.